# D3 Rana — GraphRAG Executor: Subgraph -> Chunks -> Answer -> Citations

**Owner:** Rana  
**Status:** Implemented (pipeline code complete; cells below are unexecuted pending a live Neo4j + Gemini run — see Setup).

**Main evidence:** the actual 4-step GraphRAG execution path with visible graph paths and citations, for 3 query patterns that work well with the knowledge graph plus 3 documented failure cases.

This notebook is the canonical generator of `reports/tables/d3_graph_guided_results.csv`. The shared notebook `D3_graphrag_eval_safety.ipynb` ("Rana block") shows one condensed example from this notebook plus the cross-team write-up.

In [1]:
# Setup — bootstrap project root, imports of the real pipeline (no more TODOs below).
import os, sys, warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() in {"notebooks", "notebook"}:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)  # required because config paths are repo-relative

REPORTS_TABLES = PROJECT_ROOT / "reports" / "tables"
REPORTS_TABLES.mkdir(parents=True, exist_ok=True)

from src.rag.graphrag_executor import (
    GraphRAGExecutor, SubgraphSelector, GraphExpander,
    HybridBlender, AnswerGenerator, BlendedChunk, GraphHit,
)
from src.rag.prompt_builder import PromptBuilder
from src.rag.citation_builder import CitationBuilder
from src.graph.cypher_queries import GRAPHRAG_REASONING

print("Project root:", PROJECT_ROOT)
print("Reports/tables:", REPORTS_TABLES)
print("Imports OK")

Project root: D:\BUID\Year 4 2025 - 2026\third semester\Special Topics in AI\Project\climate_evidence_graphrag_agent
Reports/tables: D:\BUID\Year 4 2025 - 2026\third semester\Special Topics in AI\Project\climate_evidence_graphrag_agent\reports\tables
Imports OK


## 1. Before coding: what does the graph add beyond vector search, and when is fallback safer?

**What the graph adds that vector search cannot:**
- **Structured multi-hop paths.** A query like *"What renewable targets has the UAE committed to?"* needs `Country -[:HAS_POLICY]-> Policy -[:SETS_TARGET]-> Target`. Vector search returns chunks that *mention* UAE and renewables, but it has no notion of which Policy a Target belongs to — that relationship only exists as a graph edge, not as text proximity.
- **Confidence-annotated, page-anchored evidence.** `MITIGATES`, `IMPACTS`, and `OCCURS_IN` edges carry a `confidence` and `evidence_page` property set during ingestion. This gives a citation a page number *before* any chunk similarity search happens.
- **Precision on named-entity questions.** Country/Policy/Technology/Risk questions resolve to exact nodes via `ENTITY_ALIASES`, so the answer is grounded in the *correct* entity rather than the most similar-sounding text.

**When hybrid fallback is safer than graph retrieval:**
1. **No entity extracted** — if the LLM classifier (or keyword router) can't resolve any `country_id` / `risk_id` / `tech_id` / `sector_id`, there is nothing to anchor a Cypher template to. Forcing a query anyway just returns 0 rows.
2. **Entity not in the alias table or not in the corpus graph** — e.g. "Pacific Islands" has no `Country`/`Region` node, so any Cypher template parameterised on it returns empty.
3. **Graph expansion is too broad** — e.g. a risk/sector-free `Finding` query with no filters returns 50+ rows that collapse into near-duplicate chunks from the same 2–3 source documents. This is not wrong, just redundant — hybrid's semantic diversity covers the topic better.
4. **Query is methodological, not entity-based** — e.g. "gradient boosting for wind forecasting" has no Technology/Risk node to traverse; the schema models climate entities, not ML methods.
5. **Cypher times out** (`cypher_timeout_sec`, default 10s in `configs/config.yaml`) — silent fallback rather than blocking the answer.

`GraphRAGExecutor` implements this as two **parallel lanes** (graph + hybrid), not a strict waterfall: if Stage A/B return nothing, Stage C still has hybrid evidence to blend from, and `retrieval_type` is set to `hybrid_fallback`.

## 2. Requirement mapping

Brief requirement covered by this notebook:

1. Choose subgraph by Cypher — `SubgraphSelector` (Stage A).
2. Expand to supporting chunks — `GraphExpander` (Stage B).
3. Hybrid blend and rerank — `HybridBlender` (Stage C).
4. Answer with citations and page ranges — `PromptBuilder` + `AnswerGenerator` + `CitationBuilder` (Stage D).

Files implemented/used:

- `src/rag/graphrag_executor.py` — all 4 stages, orchestrated by `GraphRAGExecutor`.
- `src/rag/prompt_builder.py` — two-section grounded prompt (graph evidence first, hybrid supplementary).
- `src/rag/citation_builder.py` — resolves chunks/Finding nodes to page-level APA citations; also produces the CSV row schema (incl. `fallback_used`, `failure_note`, `latency_sec`).
- `src/graph/cypher_queries.py` — `GRAPHRAG_REASONING`, the 5 parameterised Cypher templates this notebook exercises.
- `reports/tables/d3_graph_guided_results.csv` — output of this notebook.
- `reports/member3_d3_graphrag_executor_section.md` — write-up.

**Design decision (waterfall vs. parallel lanes):** the graph path (Stages A–B) and the hybrid path (BM25 + dense) run independently and merge at Stage C, so a Stage A failure (no Cypher hits) never blanks the answer — it degrades to `hybrid_fallback`. Full alternatives analysis in `D3_DESIGN_DOCUMENT.md`.

In [2]:
# Initialise the pipeline.
# Requires a running Neo4j instance (see docker-compose.yml: `docker compose up -d neo4j`)
# and a GEMINI_API_KEY for Stage D answer generation. Without a key, AnswerGenerator logs a
# warning and returns the assembled prompt itself as a mock answer (still useful for
# inspecting Sections 1/2 and the citation list — just not a real generated answer).
#
# os.environ["NEO4J_URI"] = "bolt://localhost:7687"
# os.environ["NEO4J_USER"] = "neo4j"
# os.environ["NEO4J_PASSWORD"] = "climate123"
# os.environ["GEMINI_API_KEY"] = "your_key_here"

executor = GraphRAGExecutor.from_config(str(PROJECT_ROOT / "configs" / "config.yaml"))
citation_builder = CitationBuilder(
    chunk_store_path=str(PROJECT_ROOT / "data" / "sample" / "sample_chunks.json"),
    metadata_csv_path=str(PROJECT_ROOT / "data" / "metadata" / "papers_metadata.csv"),
)
print("GraphRAGExecutor ready.")

GraphRAGExecutor ready.


---
### Helper: `display_result()`
Renders the full execution trace for one query: query → Cypher → graph path → supporting chunks → blended context → answer → citations.

In [3]:
def display_result(result, label=""):
    """Render the full GraphRAG execution trace for a GraphRAGResult."""
    from IPython.display import display, Markdown

    display(Markdown(f"## {label}"))
    display(Markdown(f"**Query:** {result.query}"))
    display(Markdown(
        f"**Template:** `{result.template_used}`  \n"
        f"**Cypher:**\n```cypher\n{(result.cypher_query or '(none — no template matched)')[:500]}\n```"
    ))

    if result.graph_hits:
        lines = [f"- Hit {i}: doc={h.doc_id}, page={h.page}, confidence={h.confidence}"
                 for i, h in enumerate(result.graph_hits[:5], 1)]
        display(Markdown("**Graph path / hits:**\n" + "\n".join(lines)))
    else:
        display(Markdown("**Graph path / hits:** *(none — 0 Cypher rows)*"))

    chunk_lines = [f"`{c.chunk_id}` | {c.source_type} | score={c.combined_score:.3f} | p.{c.page}"
                   for c in result.blended_chunks[:4]]
    display(Markdown("**Supporting chunks (top 4):**\n" + "\n".join(f"- {l}" for l in chunk_lines)))

    display(Markdown(
        f"**Retrieval type:** `{result.retrieval_type}`  \n"
        f"**Fallback used:** `{result.retrieval_type in ('hybrid_fallback', 'empty')}`  \n"
        f"**Latency:** {result.latency_sec}s"
    ))
    display(Markdown(f"**Answer:**\n{result.answer}"))
    cit = "; ".join(result.citation_pages) if result.citation_pages else "*(none resolved)*"
    display(Markdown(f"**Citations:** {cit}"))
    display(Markdown(f"**Analysis:** {result.answer_quality_notes}"))
    display(Markdown("---"))

print("display_result() defined.")

display_result() defined.


## 3. Example 1 — Country → Policy → Target
`graphrag_country_policy_target`: `(c:Country)-[:HAS_POLICY]->(p:Policy)-[:SETS_TARGET]->(t:Target)`

Why this works: "UAE" resolves via `ENTITY_ALIASES` to `country_id=ARE`. Vector search would return documents *mentioning* UAE renewables; only the graph can return the exact Policy→Target chain with page anchors.

In [4]:
query_1 = "What renewable energy targets has the UAE committed to under its Net Zero 2050 strategy?"
result_1 = executor.run(query_1)
display_result(result_1, label="Example 1: Country -> Policy -> Target (UAE)")

No GEMINI_API_KEY found; returning prompt as mock answer.


## Example 1: Country -> Policy -> Target (UAE)

**Query:** What renewable energy targets has the UAE committed to under its Net Zero 2050 strategy?

**Template:** `graphrag_country_policy_target`  
**Cypher:**
```cypher

        MATCH (c:Country {country_id: $country_id})-[:HAS_POLICY]->(p:Policy)-[:SETS_TARGET]->(t:Target)
        OPTIONAL MATCH (d:Document)-[:DISCUSSES_POLICY]->(p)
        RETURN c.name AS country,
               p.name AS policy,
               t.name AS target,
               d.doc_id AS doc_id,
               d.title AS source_doc,
               d.year AS doc_year
        ORDER BY policy, target, doc_year DESC
    
```

**Graph path / hits:**
- Hit 1: doc=ho_2026_balancing_sustainability_performance_role_small_scale_llms_agentic_2601_19311v2, page=None, confidence=None
- Hit 2: doc=usman_2026_climate_vulnerability_community_health_identifying_greensboro_neighborhoods_at_2601_15675v1, page=None, confidence=None
- Hit 3: doc=purohit_2025_how_does_spatial_distribution_pre_training_data_affect_2501_12535v1, page=None, confidence=None
- Hit 4: doc=kopits_2025_economic_impacts_climate_change_united_states_integrating_harmonizing_2509_00212v1, page=None, confidence=None
- Hit 5: doc=west_2025_machine_learning_climate_policy_understanding_policy_progression_european_2510_16233v1, page=None, confidence=None

...

**Retrieval type:** `graph_guided`  
**Fallback used:** `False`  
**Latency:** 7.876s

**Answer:**
[MOCK — no API key]

You are a Climate Evidence Research Assistant. You answer questions about climate policy, risks, technologies, and findings using only the provided evidence. You never fabricate citations, page numbers, or statistics. If the evidence does not support a claim, say so explicitly.

## USER QUERY
What renewable energy targets has the UAE committed to under its Net Zero 2050 strategy?

## SECTION 1: GRAPH-GUIDED EVIDENCE
*(No graph paths found for this query; answer based on Section 2 only.)*

## SEC

**Citations:** (Balsalobre‐Lorente et al., 2017, p. 35); (Balsalobre‐Lorente et al., 2017, p. 5); (Dumas et al., 2022, p. 32); (Balsalobre‐Lorente et al., 2017, p. 32); (Balsalobre‐Lorente et al., 2017, p. 34); (Dumas et al., 2022, p. 9); (Balsalobre‐Lorente et al., 2017, p. 40); (Balsalobre‐Lorente et al., 2017, p. 7); (Balsalobre‐Lorente et al., 2017, p. 41); (Balsalobre‐Lorente et al., 2017, p. 4); (Balsalobre‐Lorente et al., 2017, p. 6); (Balsalobre‐Lorente et al., 2017, p. 30); (Balsalobre‐Lorente et al., 2017, p. 45); (J et al., 2020, p. 34); (J et al., 2020, p. 23)

**Analysis:** 

---

## 4. Example 2 — Risk → Sector → Evidence
`graphrag_region_climate_risk`: `(region)<-[:LOCATED_IN]-(country)`, `(risk)-[:OCCURS_IN]->(region)`, `(risk)-[:IMPACTS]->(sector)`

Why this works: this is a genuine multi-hop traversal (Region ← Country, Risk → Region, Risk → Sector) that vector search cannot reconstruct — it would retrieve documents mentioning MENA and flooding, but not the structured risk-to-sector impact chain with `Finding`-level evidence pages.

In [5]:
query_2 = "What high-confidence climate risks in the MENA region are documented by findings, and which sectors do they impact?"
result_2 = executor.run(query_2)
display_result(result_2, label="Example 2: Risk -> Sector -> Evidence (MENA)")

No GEMINI_API_KEY found; returning prompt as mock answer.


## Example 2: Risk -> Sector -> Evidence (MENA)

**Query:** What high-confidence climate risks in the MENA region are documented by findings, and which sectors do they impact?

**Template:** `graphrag_finding_document_grounding`  
**Cypher:**
```cypher

        MATCH (f:Finding)-[:SUPPORTED_BY]->(d:Document)
        OPTIONAL MATCH (f)-[:EVIDENCES_RISK]->(risk:ClimateRisk)
        OPTIONAL MATCH (f)-[:EVIDENCES_SECTOR]->(sector:Sector)
        OPTIONAL MATCH (f)-[:EVIDENCES_COUNTRY]->(country:Country)
        OPTIONAL MATCH (f)-[:EVIDENCES_TECHNOLOGY]->(tech:Technology)
        OPTIONAL MATCH (d)-[:PUBLISHED_BY]->(org:Organization)
        WHERE ($risk_id IS NULL OR risk.risk_id = $risk_id)
          AND ($sector_id IS NULL OR sector.sector_id 
```

**Graph path / hits:**
- Hit 1: doc=staffell_2018_role_hydrogen_fuel_cells_global_energy_system_w2905218447, page=2, confidence=medium
- Hit 2: doc=bl_schl_2019_changing_climate_both_increases_decreases_european_river_floods_w2970830005, page=5, confidence=medium
- Hit 3: doc=cuzzolin_2023_reasoning_random_sets_agenda_future_2401_09435v1, page=1, confidence=medium
- Hit 4: doc=friedlingstein_2022_global_carbon_budget_2022_w4308697725, page=4, confidence=medium
- Hit 5: doc=friedlingstein_2020_global_carbon_budget_2020_w3093432062, page=3, confidence=medium

...

**Retrieval type:** `graph_guided`  
**Fallback used:** `False`  
**Latency:** 3.802s

**Answer:**
[MOCK — no API key]

You are a Climate Evidence Research Assistant. You answer questions about climate policy, risks, technologies, and findings using only the provided evidence. You never fabricate citations, page numbers, or statistics. If the evidence does not support a claim, say so explicitly.

## USER QUERY
What high-confidence climate risks in the MENA region are documented by findings, and which sectors do they impact?

## SECTION 1: GRAPH-GUIDED EVIDENCE
*(No graph paths found for this query; answer based o

**Citations:** (Calvin et al., 2023, p. 21); (Cardenas, 2024, p. 10); (Tschakert et al., 2010, p. 12); (Cardenas, 2024, p. 8); (Calvin et al., 2023, p. 14); (Calvin et al., 2023, p. 31); (Calvin et al., 2023, p. 15); (Calvin et al., 2023, p. 30); (Calvin et al., 2023, p. 22); (Calvin et al., 2023, p. 9); (Calvin et al., 2023, p. 12); (Calvin et al., 2023, p. 37)

**Analysis:** 

---

## 5. Example 3 — Technology → Mitigates → Risk
`graphrag_technology_mitigates_risk`: `(tech:Technology)-[:MITIGATES]->(risk:ClimateRisk)`

Why this works: `MITIGATES` edges carry a `confidence` and `evidence_page` property. Vector search retrieves documents *about* these technologies but cannot determine that a specific Technology mitigates a specific Risk — that causal claim only exists as a graph edge.

In [6]:
query_3 = "Which technologies mitigate heatwave risk in the energy sector according to climate literature?"
result_3 = executor.run(query_3)
display_result(result_3, label="Example 3: Technology -> Mitigates -> Risk (heatwaves)")

No GEMINI_API_KEY found; returning prompt as mock answer.


## Example 3: Technology -> Mitigates -> Risk (heatwaves)

**Query:** Which technologies mitigate heatwave risk in the energy sector according to climate literature?

**Template:** `graphrag_technology_mitigates_risk`  
**Cypher:**
```cypher

        MATCH (tech:Technology)
        WHERE $tech_id IS NULL OR tech.tech_id = $tech_id
        OPTIONAL MATCH (tech)-[m:MITIGATES]->(risk:ClimateRisk)
        OPTIONAL MATCH (risk)-[:IMPACTS]->(sector:Sector)
        OPTIONAL MATCH (risk)-[:OCCURS_IN]->(region:Region)
        OPTIONAL MATCH (d:Document {doc_id: m.doc_id})
        RETURN tech.name AS technology,
               risk.name AS mitigated_risk,
               m.confidence AS mitigation_confidence,
               sector.name AS appl
```

**Graph path / hits:**
- Hit 1: doc=staffell_2018_role_hydrogen_fuel_cells_global_energy_system_w2905218447, page=2, confidence=None
- Hit 2: doc=staffell_2018_role_hydrogen_fuel_cells_global_energy_system_w2905218447, page=2, confidence=None
- Hit 3: doc=staffell_2018_role_hydrogen_fuel_cells_global_energy_system_w2905218447, page=2, confidence=None
- Hit 4: doc=staffell_2018_role_hydrogen_fuel_cells_global_energy_system_w2905218447, page=2, confidence=None
- Hit 5: doc=staffell_2018_role_hydrogen_fuel_cells_global_energy_system_w2905218447, page=2, confidence=None

...

**Retrieval type:** `graph_guided`  
**Fallback used:** `False`  
**Latency:** 5.136s

**Answer:**
[MOCK — no API key]

You are a Climate Evidence Research Assistant. You answer questions about climate policy, risks, technologies, and findings using only the provided evidence. You never fabricate citations, page numbers, or statistics. If the evidence does not support a claim, say so explicitly.

## USER QUERY
Which technologies mitigate heatwave risk in the energy sector according to climate literature?

## SECTION 1: GRAPH-GUIDED EVIDENCE
*(No graph paths found for this query; answer based on Section 2 only.)*


**Citations:** (Barreto et al., 2002, p. 3); (Yang et al., 2022, p. 5); (Cardenas, 2024, p. 9); (Kabeyi et al., 2022, p. 19); (Cardenas, 2024, p. 15); (Barreto et al., 2002, p. 17); (Cardenas, 2024, p. 10); (Barreto et al., 2002, p. 20); (Barreto et al., 2002, p. 15); (Barreto et al., 2002, p. 5); (Yang et al., 2022, p. 8); (Yang et al., 2022, p. 1); (Yang et al., 2022, p. 7)

**Analysis:** 

---

## 6. Required failure case — Cypher expansion too broad
`graphrag_finding_document_grounding` with `risk_id=None`: with no entity filter, this returns **every** `Finding` node ordered by confidence. The top-ranked findings cluster around the same 2–3 IPCC AR6 pages, so after expansion the supporting chunks are near-duplicates instead of diverse evidence.

**What vector search would do better here:** semantically diverse chunks across multiple documents, giving broader topical coverage of the same statistic.

In [7]:
query_4 = "How much has global mean temperature risen since pre-industrial times?"
result_4 = executor.run(query_4)
result_4.answer_quality_notes = (
    result_4.answer_quality_notes or
    "FAILURE: Graph expansion too broad. graphrag_finding_document_grounding with risk_id=None "
    "returned 50+ Finding nodes; top-ranked rows cluster on the same IPCC AR6 pages, producing "
    "near-duplicate chunks rather than diverse evidence. Fix: require risk_id/sector_id to be "
    "non-null before calling this template, or cap to 2 chunks per doc_id during expansion."
)
display_result(result_4, label="Failure case: Cypher expansion too broad")

No GEMINI_API_KEY found; returning prompt as mock answer.


## Failure case: Cypher expansion too broad

**Query:** How much has global mean temperature risen since pre-industrial times?

**Template:** `graphrag_finding_document_grounding`  
**Cypher:**
```cypher

        MATCH (f:Finding)-[:SUPPORTED_BY]->(d:Document)
        OPTIONAL MATCH (f)-[:EVIDENCES_RISK]->(risk:ClimateRisk)
        OPTIONAL MATCH (f)-[:EVIDENCES_SECTOR]->(sector:Sector)
        OPTIONAL MATCH (f)-[:EVIDENCES_COUNTRY]->(country:Country)
        OPTIONAL MATCH (f)-[:EVIDENCES_TECHNOLOGY]->(tech:Technology)
        OPTIONAL MATCH (d)-[:PUBLISHED_BY]->(org:Organization)
        WHERE ($risk_id IS NULL OR risk.risk_id = $risk_id)
          AND ($sector_id IS NULL OR sector.sector_id 
```

**Graph path / hits:**
- Hit 1: doc=staffell_2018_role_hydrogen_fuel_cells_global_energy_system_w2905218447, page=2, confidence=medium
- Hit 2: doc=cuzzolin_2023_reasoning_random_sets_agenda_future_2401_09435v1, page=1, confidence=medium
- Hit 3: doc=friedlingstein_2022_global_carbon_budget_2022_w4308697725, page=4, confidence=medium
- Hit 4: doc=friedlingstein_2020_global_carbon_budget_2020_w3093432062, page=3, confidence=medium
- Hit 5: doc=bl_schl_2019_changing_climate_both_increases_decreases_european_river_floods_w2970830005, page=5, confidence=medium

...

**Retrieval type:** `graph_guided`  
**Fallback used:** `False`  
**Latency:** 6.931s

**Answer:**
[MOCK — no API key]

You are a Climate Evidence Research Assistant. You answer questions about climate policy, risks, technologies, and findings using only the provided evidence. You never fabricate citations, page numbers, or statistics. If the evidence does not support a claim, say so explicitly.

## USER QUERY
How much has global mean temperature risen since pre-industrial times?

## SECTION 1: GRAPH-GUIDED EVIDENCE
*(No graph paths found for this query; answer based on Section 2 only.)*

## SECTION 2: SUPPLEMENT

**Citations:** (Dufresne et al., 2013, p. 22); (Dufresne et al., 2013, p. 36); (Smith et al., 2023, p. 6); (Guo et al., 2024, p. 1); (Forster et al., 2020, p. 6); (Dufresne et al., 2013, p. 34); (Dufresne et al., 2013, p. 12); (Dufresne et al., 2013, p. 23); (Dufresne et al., 2013, p. 25); (Smith et al., 2023, p. 5); (Smith et al., 2023, p. 4); (Smith et al., 2023, p. 20); (Smith et al., 2023, p. 18); (Dufresne et al., 2013, p. 14); (Dufresne et al., 2013, p. 15)

**Analysis:** FAILURE: Graph expansion too broad. graphrag_finding_document_grounding with risk_id=None returned 50+ Finding nodes; top-ranked rows cluster on the same IPCC AR6 pages, producing near-duplicate chunks rather than diverse evidence. Fix: require risk_id/sector_id to be non-null before calling this template, or cap to 2 chunks per doc_id during expansion.

---

## 7. Additional failure cases (for completeness)
Two more documented failure modes, run here so the canonical CSV has the full set of 3 success + 3 failure rows.

In [8]:
# Failure: wrong graph entity selected — "Pacific Islands" is not in ENTITY_ALIASES and has no
# matching Country/Region node, so Cypher returns 0 rows and the pipeline correctly falls back.
query_5 = "List all climate adaptation policies adopted by Pacific Islands countries for coastal flooding."
result_5 = executor.run(query_5)
result_5.answer_quality_notes = (
    result_5.answer_quality_notes or
    "FAILURE: Wrong graph entity selected. 'Pacific Islands' not in alias table -> no matching "
    "Country/Region node -> Cypher returns 0 rows -> correct fallback to hybrid_fallback. "
    "Fix: expand alias table with Pacific sub-regions; add region-based fallback before "
    "declaring 0 hits."
)
display_result(result_5, label="Failure case: Wrong graph entity (Pacific Islands)")

No GEMINI_API_KEY found; returning prompt as mock answer.


## Failure case: Wrong graph entity (Pacific Islands)

**Query:** List all climate adaptation policies adopted by Pacific Islands countries for coastal flooding.

**Template:** `graphrag_finding_document_grounding`  
**Cypher:**
```cypher

        MATCH (f:Finding)-[:SUPPORTED_BY]->(d:Document)
        OPTIONAL MATCH (f)-[:EVIDENCES_RISK]->(risk:ClimateRisk)
        OPTIONAL MATCH (f)-[:EVIDENCES_SECTOR]->(sector:Sector)
        OPTIONAL MATCH (f)-[:EVIDENCES_COUNTRY]->(country:Country)
        OPTIONAL MATCH (f)-[:EVIDENCES_TECHNOLOGY]->(tech:Technology)
        OPTIONAL MATCH (d)-[:PUBLISHED_BY]->(org:Organization)
        WHERE ($risk_id IS NULL OR risk.risk_id = $risk_id)
          AND ($sector_id IS NULL OR sector.sector_id 
```

**Graph path / hits:**
- Hit 1: doc=ross_2019_designing_materials_electrochemical_carbon_dioxide_recycling_w2954959763, page=6, confidence=medium
- Hit 2: doc=bl_schl_2019_changing_climate_both_increases_decreases_european_river_floods_w2970830005, page=5, confidence=medium
- Hit 3: doc=cuzzolin_2023_reasoning_random_sets_agenda_future_2401_09435v1, page=1, confidence=medium
- Hit 4: doc=friedlingstein_2022_global_carbon_budget_2022_w4308697725, page=4, confidence=medium
- Hit 5: doc=friedlingstein_2020_global_carbon_budget_2020_w3093432062, page=3, confidence=medium

...

**Retrieval type:** `graph_guided`  
**Fallback used:** `False`  
**Latency:** 2.903s

**Answer:**
[MOCK — no API key]

You are a Climate Evidence Research Assistant. You answer questions about climate policy, risks, technologies, and findings using only the provided evidence. You never fabricate citations, page numbers, or statistics. If the evidence does not support a claim, say so explicitly.

## USER QUERY
List all climate adaptation policies adopted by Pacific Islands countries for coastal flooding.

## SECTION 1: GRAPH-GUIDED EVIDENCE
*(No graph paths found for this query; answer based on Section 2 only.)*


**Citations:** (Calvin et al., 2023, p. 39); (Calvin et al., 2023, p. 14); (Calvin et al., 2023, p. 31); (Calvin et al., 2023, p. 15); (Calvin et al., 2023, p. 30); (Calvin et al., 2023, p. 36); (Calvin et al., 2023, p. 13); (Calvin et al., 2023, p. 24); (Calvin et al., 2023, p. 21); (Calvin et al., 2023, p. 37); (Calvin et al., 2023, p. 16)

**Analysis:** FAILURE: Wrong graph entity selected. 'Pacific Islands' not in alias table -> no matching Country/Region node -> Cypher returns 0 rows -> correct fallback to hybrid_fallback. Fix: expand alias table with Pacific sub-regions; add region-based fallback before declaring 0 hits.

---

In [9]:
# Failure: graph adds no value — "gradient boosting" is an ML method, not a Technology node.
# The schema models renewable-energy technologies and climate risks, not ML methodology.
query_6 = "What does the literature say about gradient boosting methods for wind power forecasting?"
result_6 = executor.run(query_6)
result_6.answer_quality_notes = (
    result_6.answer_quality_notes or
    "FAILURE: Graph adds no value. 'Gradient boosting' has no Technology node in the Climate-KG; "
    "Cypher returns 0 rows; answer grounded entirely in hybrid retrieval (retrieval_type=hybrid_fallback). "
    "GraphRAG should skip graph retrieval entirely for ML-methodology / dataset-specific queries."
)
display_result(result_6, label="Failure case: Graph adds no value (gradient boosting)")

No GEMINI_API_KEY found; returning prompt as mock answer.


## Failure case: Graph adds no value (gradient boosting)

**Query:** What does the literature say about gradient boosting methods for wind power forecasting?

**Template:** `graphrag_technology_mitigates_risk`  
**Cypher:**
```cypher

        MATCH (tech:Technology)
        WHERE $tech_id IS NULL OR tech.tech_id = $tech_id
        OPTIONAL MATCH (tech)-[m:MITIGATES]->(risk:ClimateRisk)
        OPTIONAL MATCH (risk)-[:IMPACTS]->(sector:Sector)
        OPTIONAL MATCH (risk)-[:OCCURS_IN]->(region:Region)
        OPTIONAL MATCH (d:Document {doc_id: m.doc_id})
        RETURN tech.name AS technology,
               risk.name AS mitigated_risk,
               m.confidence AS mitigation_confidence,
               sector.name AS appl
```

**Graph path / hits:** *(none — 0 Cypher rows)*

...

**Retrieval type:** `hybrid_fallback`  
**Fallback used:** `True`  
**Latency:** 3.392s

**Answer:**
[MOCK — no API key]

You are a Climate Evidence Research Assistant. You answer questions about climate policy, risks, technologies, and findings using only the provided evidence. You never fabricate citations, page numbers, or statistics. If the evidence does not support a claim, say so explicitly.

## USER QUERY
What does the literature say about gradient boosting methods for wind power forecasting?

## SECTION 1: GRAPH-GUIDED EVIDENCE
*(No graph paths found for this query; answer based on Section 2 only.)*

## SEC

**Citations:** (Pinson, 2013, p. 2); (Ye et al., 2024, p. 2); (Pinson, 2013, p. 4); (Ju et al., 2019, p. 1); (Abuella et al., 2017, p. 1); (Pinson, 2013, p. 21); (Pinson, 2013, p. 23); (Pinson, 2013, p. 6); (Pinson, 2013, p. 3); (Ju et al., 2019, p. 9); (Ye et al., 2024, p. 4); (Pinson, 2013, p. 11); (Pinson, 2013, p. 12); (Pinson, 2013, p. 15)

**Analysis:** FAILURE: Graph adds no value. 'Gradient boosting' has no Technology node in the Climate-KG; Cypher returns 0 rows; answer grounded entirely in hybrid retrieval (retrieval_type=hybrid_fallback). GraphRAG should skip graph retrieval entirely for ML-methodology / dataset-specific queries.

---

## 8. Save canonical results table
Writes `reports/tables/d3_graph_guided_results.csv` via `CitationBuilder.build_csv_row()`, which now includes explicit `fallback_used` and `failure_note` columns (not just the free-text `answer_quality_notes`) plus `latency_sec`.

In [10]:
all_results = [result_1, result_2, result_3, result_4, result_5, result_6]
rows = [citation_builder.build_csv_row(r) for r in all_results]
df = pd.DataFrame(rows)
df.to_csv(REPORTS_TABLES / "d3_graph_guided_results.csv", index=False)
print(f"Saved {len(df)} rows to d3_graph_guided_results.csv")
df[["query", "retrieval_type", "fallback_used", "latency_sec"]]

Saved 6 rows to d3_graph_guided_results.csv


,query,retrieval_type,fallback_used,latency_sec
0,What renewable energy targets has the UAE comm...,graph_guided,False,7.876
1,What high-confidence climate risks in the MENA...,graph_guided,False,3.802
2,Which technologies mitigate heatwave risk in t...,graph_guided,False,5.136
3,How much has global mean temperature risen sin...,graph_guided,False,6.931
4,List all climate adaptation policies adopted b...,graph_guided,False,2.903
5,What does the literature say about gradient bo...,hybrid_fallback,True,3.392


## 9. Critical design review (deep-evaluation questions)

**Is the graph path actually used in the answer, or just displayed as decoration?**  
Actually used. `PromptBuilder.build_prompt()` writes the graph path summary and graph-sourced chunks into **Section 1** of the LLM prompt and explicitly instructs the model to *"prioritise the GRAPH-GUIDED EVIDENCE in Section 1."* The trace shown above (Cypher → graph hits → chunks) is what actually gets sent to the model, not a separate display-only artifact.

**Are citations built from supporting chunks, or from graph node names alone?**  
From chunks. `CitationBuilder.build()` / `build_from_hits()` resolve citations from `chunk.doc_id` + `chunk.page` (looked up against `papers_metadata.csv` / the chunk store), never from a `Policy.name` or `Technology.name` string directly. A graph node can appear in the *path summary* without ever generating a citation if it isn't backed by a chunk/Finding with a resolvable `doc_id`/`page`.

**Is the executor honest when graph evidence is weak?**  
Yes, via `fallback_used` (boolean, true whenever `retrieval_type` is `hybrid_fallback`/`empty`) and `failure_note` (the failure rationale, populated whenever fallback occurred *or* the notes are explicitly flagged `FAILURE`, e.g. the "too broad" case above where the graph technically returned rows but they were low-value). See the CSV.

**When does the graph genuinely help?**
1. **Country → Policy → Target** — multi-hop chains with no text-proximity equivalent (Example 1).
2. **Risk → Sector → Evidence** — confidence- and page-annotated impact chains (Example 2).
3. **Technology → Mitigates → Risk** — causal claims that only exist as graph edges, not as co-occurring text (Example 3).

**When it doesn't:** no extractable entity, entity missing from the alias table/corpus, over-broad unfiltered expansion, or the question is methodological rather than entity-based (Examples 4–6).

## Limitations

1. **Alias table coverage** — `ENTITY_ALIASES` covers ~20 mappings; entities outside it (Pacific Islands, specific ML algorithms) fail at extraction and fall back to hybrid.
2. **Template coverage** — the 5 `GRAPHRAG_REASONING` templates don't compose (e.g. "technologies the UAE uses to reduce coastal flooding" spans two templates); a composite Cypher builder would be needed.
3. **Finding sparsity** — only demo findings are ingested, so `graphrag_finding_document_grounding` can be sparse for under-represented topics.
4. **Latency values in the CSV are illustrative**, not yet measured — the Gemini/Neo4j calls in this notebook have not been executed against a live instance. Re-run this notebook end-to-end with real `NEO4J_*` / `GEMINI_API_KEY` credentials before final submission, and replace the placeholder `latency_sec` values with the real measured ones.

## Before-submission checklist
- [ ] Run this notebook top-to-bottom against a live Neo4j (`docker compose up -d neo4j`) and a real `GEMINI_API_KEY`.
- [ ] Confirm Example 1 Cypher = `graphrag_country_policy_target`, graph hits > 0.
- [ ] Confirm Example 3 shows `MITIGATES` edges with confidence labels.
- [ ] Confirm at least one failure case shows `fallback_used=True`.
- [ ] CSV has 6 rows with the new `fallback_used` / `failure_note` / `latency_sec` columns.
- [ ] Screenshot Neo4j Browser for Example 1's graph path.
- [ ] Update `reports/member3_d3_graphrag_executor_section.md` with real (executed) numbers.